In [6]:
"""
LaDe Delivery Intelligence Pipeline
=====================================
Project: Last-mile delivery data engineering system
Dataset: LaDe - 472,419 real delivery records across 5 cities
Source: Cainiao logistics (Alibaba) - May to October 2022

Pipeline Architecture:
- Layer 1: Batch ingestion (Python + pandas + MySQL)
- Layer 2: PySpark transformation (this notebook)
- Layer 3: Star schema loading (MySQL)
- Layer 4: Business metrics (SQL)
- Layer 5: Data quality framework (Python)
- Layer 6: Orchestration (Python schedule)
- Layer 7: Streamlit dashboard (deployed)

PySpark Configuration:
- master=local[*]: runs on all available CPU cores locally
- Production equivalent: spark://cluster-master:7077
- shuffle.partitions=4: optimised for 472K rows (default 200 is overkill)

Author: Rishitha Gopagani
"""

'\nLaDe Delivery Intelligence Pipeline\n=====================================\nProject: Last-mile delivery data engineering system\nDataset: LaDe - 472,419 real delivery records across 5 cities\nSource: Cainiao logistics (Alibaba) - May to October 2022\n\nPipeline Architecture:\n- Layer 1: Batch ingestion (Python + pandas + MySQL)\n- Layer 2: PySpark transformation (this notebook)\n- Layer 3: Star schema loading (MySQL)\n- Layer 4: Business metrics (SQL)\n- Layer 5: Data quality framework (Python)\n- Layer 6: Orchestration (Python schedule)\n- Layer 7: Streamlit dashboard (deployed)\n\nPySpark Configuration:\n- master=local[*]: runs on all available CPU cores locally\n- Production equivalent: spark://cluster-master:7077\n- shuffle.partitions=4: optimised for 472K rows (default 200 is overkill)\n\nAuthor: Rishitha Gopagani\n'

In [1]:
#install Pyspark
!pip install pyspark

In [2]:
#verify installation
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder\
.appName("LaDe delivery Pipeline")\
        .master("local[*]")\
        .config("spark.sql.shuffle.partitions", 4)\
        .getOrCreate()

In [7]:
print(spark.version)
print("SparkSession created successfully")

4.0.2
SparkSession created successfully


In [8]:
#Layer 2- PySpark Transformation
#step1 : Read raw CSV into Spark DataFrame

df_raw = spark.read\
.option("header", "true")\
.csv("delivery_five_cities.csv")

print("Shape:")
print(f"Rows: {df_raw.count()}")
print(f"Columns:{len(df_raw.columns)}")
print("\nSchema:")
df_raw.printSchema()
print("\nSample rows:")
df_raw.show(3)


Shape:
Rows: 472419
Columns:15

Schema:
root
 |-- order_id: string (nullable = true)
 |-- from_dipan_id: string (nullable = true)
 |-- from_city_name: string (nullable = true)
 |-- delivery_user_id: string (nullable = true)
 |-- poi_lng: string (nullable = true)
 |-- poi_lat: string (nullable = true)
 |-- aoi_id: string (nullable = true)
 |-- typecode: string (nullable = true)
 |-- receipt_time: string (nullable = true)
 |-- receipt_lng: string (nullable = true)
 |-- receipt_lat: string (nullable = true)
 |-- sign_time: string (nullable = true)
 |-- sign_lng: string (nullable = true)
 |-- sign_lat: string (nullable = true)
 |-- ds: string (nullable = true)


Sample rows:
+--------------------+--------------------+--------------+--------------------+------------------+-------------------+--------------------+--------------------+--------------+------------------+-------------------+--------------+--------+--------+----+
|            order_id|       from_dipan_id|from_city_name|    deliv

In [9]:
from pyspark.sql.functions import col, when, unix_timestamp, round, to_timestamp, lit, regexp_replace
from pyspark.sql import functions as F

df = df_raw.drop("sign_lng", "sign_lat")
print(f"After dropping null columns: {len(df.columns)} columns remaining")

After dropping null columns: 13 columns remaining


In [10]:
# Step 2 — Cast numeric columns and map city names
df = df.withColumn("poi_lng", col("poi_lng").cast("double")) \
       .withColumn("poi_lat", col("poi_lat").cast("double")) \
       .withColumn("receipt_lng", col("receipt_lng").cast("double")) \
       .withColumn("receipt_lat", col("receipt_lat").cast("double"))

# Map Chinese city names to English
city_mapping = {
    "上海市": "Shanghai",
    "重庆市": "Chongqing",
    "杭州市": "Hangzhou",
    "吉林市": "Jilin",
    "南京市": "Nanjing"
}

df = df.withColumn("city",
    when(col("from_city_name") == "上海市", "Shanghai")
    .when(col("from_city_name") == "重庆市", "Chongqing")
    .when(col("from_city_name") == "杭州市", "Hangzhou")
    .when(col("from_city_name") == "吉林市", "Jilin")
    .when(col("from_city_name") == "南京市", "Nanjing")
    .otherwise("Unknown")
).drop("from_city_name")

print("City mapping done. Verifying:")
df.groupBy("city").count().show()

City mapping done. Verifying:
+---------+------+
|     city| count|
+---------+------+
| Hangzhou|185589|
| Shanghai|165583|
|Chongqing|121247|
+---------+------+



In [11]:
# Check what city names actually exist in raw data
df_raw.groupBy("from_city_name").count().show()

+--------------+------+
|from_city_name| count|
+--------------+------+
|        重庆市|121247|
|        杭州市|185589|
|        上海市|165583|
+--------------+------+



In [12]:
# Data insight: Delivery file contains 3 cities only
# Hangzhou: 185,589 deliveries (highest volume)
# Shanghai: 165,583 deliveries
# Chongqing: 121,247 deliveries
# Jilin and Nanjing only present in pickup dataset

In [13]:
# Step 3 — Handle nulls and parse date column
# Handle typecode nulls
df = df.withColumn("typecode",
    when(col("typecode").isNull(), "UNKNOWN")
    .otherwise(col("typecode"))
)

# Parse ds from string like "0318" to proper date "2022-03-18"
# ds format is MMDD — need to add year 2022
df = df.withColumn("delivery_date",
    to_timestamp(
        F.concat(lit("2022"),
                 F.lpad(col("ds"), 4, "0")),
        "yyyyMMdd"
    )
).drop("ds")

print("Null handling and date parsing done")
df.select("typecode", "delivery_date").show(5)

Null handling and date parsing done
+--------------------+-------------------+
|            typecode|      delivery_date|
+--------------------+-------------------+
|4602b38053ece07a9...|2022-03-18 00:00:00|
|203ac3454d75e02eb...|2022-03-18 00:00:00|
|203ac3454d75e02eb...|2022-03-18 00:00:00|
|203ac3454d75e02eb...|2022-03-18 00:00:00|
|4602b38053ece07a9...|2022-03-18 00:00:00|
+--------------------+-------------------+
only showing top 5 rows


In [14]:
# Step 4 — Parse timestamps and calculate delivery duration
# receipt_time and sign_time format is "MM-DD HH:MM:SS" — need to add year 2022

df = df.withColumn("receipt_timestamp",
    to_timestamp(
        F.concat(lit("2022-"), col("receipt_time")),
        "yyyy-MM-dd HH:mm:ss"
    )
).withColumn("sign_timestamp",
    to_timestamp(
        F.concat(lit("2022-"), col("sign_time")),
        "yyyy-MM-dd HH:mm:ss"
    )
)

# Calculate delivery duration in minutes
df = df.withColumn("delivery_duration_minutes",
    round(
        (unix_timestamp(col("sign_timestamp")) -
         unix_timestamp(col("receipt_timestamp"))) / 60,
        2
    )
)

# Drop original string time columns
df = df.drop("receipt_time", "sign_time")

print("Timestamp parsing and duration calculation done")
df.select("receipt_timestamp", "sign_timestamp", "delivery_duration_minutes").show(5)

Timestamp parsing and duration calculation done
+-------------------+-------------------+-------------------------+
|  receipt_timestamp|     sign_timestamp|delivery_duration_minutes|
+-------------------+-------------------+-------------------------+
|2022-03-18 13:35:00|2022-03-18 14:51:00|                     76.0|
|2022-03-18 08:32:00|2022-03-18 14:33:00|                    361.0|
|2022-03-18 13:02:00|2022-03-18 15:34:00|                    152.0|
|2022-03-18 12:11:00|2022-03-18 14:08:00|                    117.0|
|2022-03-18 07:28:00|2022-03-20 12:40:00|                   3192.0|
+-------------------+-------------------+-------------------------+
only showing top 5 rows


In [16]:
# Step 5 — Flag invalid records and deduplicate

# Flag rows with invalid duration
df = df.withColumn("is_valid",
    when(col("delivery_duration_minutes") < 0, False)
    .when(col("delivery_duration_minutes") > 1440, False)
    .when(col("delivery_duration_minutes").isNull(), False)
    .otherwise(True)
)

# Check validity distribution
print("Validity check:")
df.groupBy("is_valid").count().show()

# Deduplicate on order_id - keep first occurrence
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy("receipt_timestamp")
df = df.withColumn("row_num", row_number().over(window)) \
       .filter(col("row_num") == 1) \
       .drop("row_num")

print(f"Duplicate order_ids found and removed: {472419 - df.count()}")
print(f"Final row count after deduplication: {df.count()}")

Validity check:
+--------+------+
|is_valid| count|
+--------+------+
|    true|465384|
|   false|  7035|
+--------+------+

Duplicate order_ids found and removed: 0
Final row count after deduplication: 472419


In [17]:
from pyspark.sql.functions import current_timestamp

# Step 6 — Add pipeline metadata columns
df = df.withColumn("processed_at", current_timestamp()) \
       .withColumn("pipeline_version", lit("v1.0")) \
       .withColumn("source_file", lit("delivery_five_cities.csv"))

# Final schema check
print("Final schema:")
df.printSchema()
print(f"\nFinal row count: {df.count()}")
print(f"Final column count: {len(df.columns)}")

# Quick summary statistics on key metric
print("\nDelivery duration statistics (valid records only):")
df.filter(col("is_valid") == True) \
  .select("delivery_duration_minutes") \
  .describe() \
  .show()

Final schema:
root
 |-- order_id: string (nullable = true)
 |-- from_dipan_id: string (nullable = true)
 |-- delivery_user_id: string (nullable = true)
 |-- poi_lng: double (nullable = true)
 |-- poi_lat: double (nullable = true)
 |-- aoi_id: string (nullable = true)
 |-- typecode: string (nullable = true)
 |-- receipt_lng: double (nullable = true)
 |-- receipt_lat: double (nullable = true)
 |-- city: string (nullable = false)
 |-- delivery_date: timestamp (nullable = true)
 |-- receipt_timestamp: timestamp (nullable = true)
 |-- sign_timestamp: timestamp (nullable = true)
 |-- delivery_duration_minutes: double (nullable = true)
 |-- is_valid: boolean (nullable = false)
 |-- processed_at: timestamp (nullable = false)
 |-- pipeline_version: string (nullable = false)
 |-- source_file: string (nullable = false)


Final row count: 472419
Final column count: 18

Delivery duration statistics (valid records only):
+-------+-------------------------+
|summary|delivery_duration_minutes|
+------

In [18]:
# Step 7 — Write clean data to Parquet partitioned by city
df.write \
  .mode("overwrite") \
  .partitionBy("city") \
  .parquet("lade_staging_clean")

print("Clean data written to Parquet successfully")
print("Partitioned by city:")
print("- Shanghai/")
print("- Hangzhou/")
print("- Chongqing/")

Clean data written to Parquet successfully
Partitioned by city:
- Shanghai/
- Hangzhou/
- Chongqing/


In [19]:
# ============================================
# Layer 2 Summary — PySpark Transformation
# ============================================
# Input: delivery_five_cities.csv (472,419 rows, 15 columns)
# Output: lade_staging_clean/ (Parquet, partitioned by city)
#
# Transformations applied:
# 1. Dropped sign_lng, sign_lat — 100% null
# 2. Cast numeric columns from string to double
# 3. Mapped Chinese city names to English
# 4. Handled typecode nulls — replaced with UNKNOWN
# 5. Parsed ds integer to proper date
# 6. Parsed receipt_time and sign_time to timestamps
# 7. Calculated delivery_duration_minutes
# 8. Flagged 7,035 invalid records (duration > 1440 mins)
# 9. Deduplicated on order_id — 0 duplicates found
# 10. Added pipeline metadata columns
# 11. Wrote partitioned Parquet by city
#
# Data quality: 98.5% valid records (465,384 of 472,419)
# Average delivery duration: 132.76 minutes
# Production equivalent: Spark cluster + S3 Parquet storage
# ============================================
print("Layer 2 complete")

Layer 2 complete
